# Gobierto Data Probes Conclusions 🕵️‍♂️

We ran a series of python probes against the endpoints for Gobierto (`presupuestos.gobierto.es`) to evaluate direct data accessibility for municipal budgets.

## Findings

- **Gobierto API & Domains**: Direct API endpoints (`/api/v1/data/data.json`) across datasets like `presupuestos`, `budgets`, and `gobierto_budgets_data` returned HTTP 404 Not Found. Furthermore, city-specific custom domains (e.g., `presupuestos.madrid.es`) failed to resolve via DNS. The Gobierto URL structures seem to have been changed, restricted behind authentication, or the service instances have been taken down for specific cities.

You can run the cells below to reproduce these failed probes and verify the outputs yourself.


### 1. Probing the Root Gobierto Domain
This confirms the root domain `presupuestos.gobierto.es` is still technically online.

In [1]:
import requests
import urllib3
urllib3.disable_warnings()

print("Fetching presupuestos.gobierto.es...")
try:
    r = requests.get("https://presupuestos.gobierto.es", verify=False)
    print("Status:", r.status_code)
    print(r.text[:2000])
except Exception as e:
    print(e)

Fetching presupuestos.gobierto.es...
Status: 200
<!DOCTYPE html>
<html>
<head>
  <title>Gobierto Presupuestos Municipales</title>
<meta name="twitter:site" content="@gobierto">

  <link rel="stylesheet" href="/assets/gobierto_budgets/application-4a7512372dfbcc48a851faa83ec4684b540c4fb43aa3572191e9c965caffe3f2.css" media="all" data-turbolinks-track="true" />
  <script src="/assets/gobierto_budgets/application-a0f359842ac19ffc97447d8aa0ed230affa10670ecc867da9db66b9521a29a22.js" data-turbolinks-track="true"></script>
  <meta name="csrf-param" content="authenticity_token" />
<meta name="csrf-token" content="jseml_rPZgtMkc5D8RDbDD6lbRt1y7lEotP7Z-f9VmZ4APck8sfPwDm4hxcYP3rjLdAhvoGT8UIcJVNylZx0LA" />

  
  <meta name="viewport" content="width=device-width, initial-scale=1" />

  <link rel="apple-touch-icon" sizes="57x57" href="/assets/favicons/apple-touch-icon-57x57-d1e9fad350f62039afddf7391d00a982b8ddd60d801e6a1b8f08ca426a28abf5.png">
  <link rel="apple-touch-icon" sizes="60x60" href="/assets

### 3. Probing a Specific City's Internal Hierarchy (Madrid / 28079)
Looking for JSON paths, CSVs or internal API routing on the exact identifier for Madrid on Gobierto (`/municipios/28079/2023`). It returns 404.

In [5]:

url = "https://presupuestos.gobierto.es/places/28079"
try:
    r = requests.get(url, verify=False, allow_redirects=True)
    print(f"Status for {url}: {r.status_code}")
    print("URL after redirects:", r.url)
    
    apis = set(re.findall(r'(/api/[a-zA-Z0-9/\-_.]+)', r.text))
    print("API relative paths found:", apis)
    
    data_urls = set([u for u in re.findall(r'(https?://[a-zA-Z0-9./\-_]+)', r.text) if 'api' in u or 'data' in u or '.csv' in u or 'json' in u])
    print("Data absolute URLs found:", data_urls)
    
except Exception as e:
    print(e)

Status for https://presupuestos.gobierto.es/places/28079: 404
URL after redirects: https://presupuestos.gobierto.es/municipios/28079
API relative paths found: set()
Data absolute URLs found: set()


## Recommendations for Fetching Expenses Breakdown (By Area) 📊

Since our objective is finding expenses breakdown (by the area / functional & organic classification), here is how you should obtain the data:

- **City Open Data Portals**: Since the Gobierto aggregation endpoints are not immediately accessible, the most reliable source for municipal expenses breakdown is the **official Open Data Portal of each city** (e.g., `datos.madrid.es`, `opendata.bcn.cat`). They publish the executed and approved budget natively in CSV or Excel format, broken down by program and organic structure.
- **National Alternatives (Hacienda)**: The *Ministerio de Hacienda (SICAL)* publishes aggregated open datasets for all Spanish municipalities' budgets, which allows for uniform tracking across cities avoiding parsing each individual portal, although sometimes it lags in reporting compared to the direct portals.

In [6]:
import requests
import json
import urllib3
urllib3.disable_warnings()

def print_tree(node, level=0):
    indent = "  " * level
    name = node.get("name", "Unknown")
    budget = node.get("budget", 0)
    print(f"{indent}- {name} ({budget:,.2f} €)")
    if "children" in node:
        for child in node["children"]:
            print_tree(child, level + 1)

print("=== GASTOS (Economic Breakdown) ===")
url_econ = "https://presupuestos.gobierto.es/municipios/madrid/2025/G/economic.json"
try:
    r = requests.get(url_econ, verify=False)
    if r.status_code == 200:
        data = r.json()
        print_tree(data)
    else:
        print("Failed to fetch:", r.status_code)
except Exception as e:
    print(e)

print("\\n\\n=== GASTOS (Functional Breakdown) ===")
url_func = "https://presupuestos.gobierto.es/municipios/madrid/2025/G/functional.json"
try:
    r = requests.get(url_func, verify=False)
    if r.status_code == 200:
        data = r.json()
        print_tree(data)
    else:
        print("Failed to fetch:", r.status_code)
except Exception as e:
    print(e)


=== GASTOS (Economic Breakdown) ===
- economic (0.00 €)
  - Gastos en bienes corrientes y servicios (2,503,468,507.00 €)
  - Gastos de personal (1,851,600,909.00 €)
  - Transferencias corrientes (957,052,045.00 €)
  - Inversiones reales (641,201,647.00 €)
  - Transferencias de capital (186,211,197.00 €)
  - Pasivos financieros (167,759,018.00 €)
  - Activos financieros (112,200,924.00 €)
  - Gastos financieros (97,918,962.00 €)
  - Fondo de contingencia y otros imprevistos (33,679,655.00 €)
\n\n=== GASTOS (Functional Breakdown) ===
- functional (0.00 €)
  - Servicios públicos básicos (3,301,561,426.00 €)
  - Actuaciones de carácter general (916,527,194.00 €)
  - Producción de bienes públicos de carácter preferente (779,593,401.00 €)
  - Actuaciones de protección y promoción social (757,812,706.00 €)
  - Actuaciones de carácter económico (544,334,015.00 €)
  - Deuda pública (251,264,122.00 €)
